[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aims-foundations/torch_measure/blob/main/tutorials/generalizability_theory.ipynb)

# Generalizability Theory: ANOVA and Hierarchical Bayesian G-Studies

**Generalizability theory (G-theory)** decomposes an observed score into variance
components: how much comes from the object of measurement (e.g. the model being
evaluated), how much comes from the measurement conditions (e.g. the benchmark
items), and how much comes from their interaction. This tells you not just a
benchmark score, but how *reliable* that score is, and how many items you would
need for a target reliability.

`torch_measure` implements two estimators for the two-way crossed (model x item)
design:

- `variance_components()` -- classical ANOVA (Method of Moments), fast, but
  assumes Gaussian, constant-variance errors.
- `bayesian_variance_components()` -- a hierarchical Bayesian model with a
  Bernoulli likelihood, fit via HMC/NUTS, appropriate for binary (right/wrong)
  benchmark responses where the ANOVA assumption does not hold.

This tutorial walks through both, on synthetic data with known ground truth,
then shows how their results compare on real benchmark data.

## 1. Setup

In [ ]:
try:
    import google.colab
    !git clone https://github.com/aims-foundations/torch_measure.git
    !pip install -e "torch_measure[data]"
except ImportError:
    pass  # Already installed locally

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch_measure.metrics.generalizability import (
    variance_components,
    bayesian_variance_components,
    g_coefficient,
    d_study,
)

## 2. Why Two Estimators?

The classical G-study estimates variance components by matching observed and
expected ANOVA mean squares, which is derived assuming every response has the
same amount of noise (homoscedastic Gaussian errors). For binary right/wrong
responses, that is not true: `Var(Y) = p(1-p)` depends on how easy or hard the
item is for that particular model. Items near 50% accuracy carry much more
noise than items every model gets right or wrong.

This mismatch can push ANOVA's variance estimates negative (which then get
clamped to zero, introducing bias), and it has no principled way to propagate
uncertainty. `bayesian_variance_components()` fits the same underlying
model x item design but with a likelihood that matches the actual binary
data-generating process:

```
mu ~ Normal(0, 2)
sigma_p, sigma_i, sigma_pi ~ HalfNormal(1)
z_p, z_i, z_pi ~ Normal(0, 1)        # non-centered parameterization
logit P(Y_ij = 1) = mu + sigma_p * z_p[i] - sigma_i * z_i[j] + sigma_pi * z_pi[i, j]
Y_ij ~ Bernoulli(sigmoid(...))
```

Both functions return the same four top-level keys (`subject`, `item`,
`subject_item`, `residual`), so `g_coefficient()` and `d_study()` work
unchanged on either estimator's output.

## 3. Generate Synthetic Data with Known Ground Truth

We generate binary responses from known variance components, so we can check
that both estimators recover the true values. This mirrors the synthetic
validation used before either estimator was trusted on real data.

In [ ]:
torch.manual_seed(42)
rng = np.random.default_rng(42)

n_subjects, n_items = 20, 30

# True parameters (variance scale)
sigma_p_true, sigma_i_true, sigma_pi_true = 0.5, 0.3, 0.4
true_vc = {
    "sigma2_p": sigma_p_true ** 2,
    "sigma2_i": sigma_i_true ** 2,
    "sigma2_pi": sigma_pi_true ** 2,
}

theta = rng.normal(0, sigma_p_true, size=n_subjects)          # model abilities
beta = rng.normal(0, sigma_i_true, size=n_items)               # item difficulties
gamma = rng.normal(0, sigma_pi_true, size=(n_subjects, n_items))  # interactions

logit = theta[:, None] - beta[None, :] + gamma
prob = 1.0 / (1.0 + np.exp(-logit))
X = (rng.uniform(size=(n_subjects, n_items)) < prob).astype(int)

print(f"Design: {n_subjects} subjects x {n_items} items")
print(f"True sigma2_p={true_vc['sigma2_p']:.3f}, sigma2_i={true_vc['sigma2_i']:.3f}, "
      f"sigma2_pi={true_vc['sigma2_pi']:.3f}")
print(f"Overall accuracy: {X.mean():.3f}")

# Long-form DataFrame, matching torch_measure's expected schema
rows = []
for i in range(n_subjects):
    for j in range(n_items):
        rows.append({
            "subject_id": f"model_{i:03d}",
            "item_id": f"item_{j:03d}",
            "trial": 1,
            "response": int(X[i, j]),
        })
responses = pd.DataFrame(rows)
responses.head()

## 4. Classical G-Study (ANOVA)

`variance_components()` runs in milliseconds -- it is a closed-form
Method-of-Moments solve, no iterative fitting.

In [ ]:
vc_anova = variance_components(responses)

print("ANOVA variance components (observed 0/1 scale):")
for key in ["subject", "item", "subject_item"]:
    print(f"  {key:<15} estimate={vc_anova[key]:.4f}")
print(f"  residual        estimate={vc_anova['residual']:.4f}   "
      f"(unidentifiable at n_r=1: {not vc_anova['identifiable']['residual']})")

**Notice these numbers do not look anything like `true_vc` from Section 3**
(e.g. the interaction term above is usually the largest of the three, even
though the true interaction variance was the *smallest* of the three
parameters used to generate the data). This is not ANOVA failing to
estimate correctly -- it is not estimating the same quantity at all.

`responses` was generated on a **latent logit scale**: `sigma_p`, `sigma_i`,
`sigma_pi` are the standard deviations of effects that get passed through a
sigmoid before becoming a 0/1 outcome (exactly what
`bayesian_variance_components()` models). `variance_components()` has no
notion of that latent scale -- it estimates variance directly on the
observed 0/1 responses, which is a bounded, nonlinear transform of the
latent scale. Comparing an observed-scale number to a latent-scale true
value is comparing two different quantities, not the same quantity
measured well or badly.

This is the same phenomenon Section 8 shows on real benchmark data: ANOVA
and the Bayesian model can disagree sharply on which component looks
largest, because they are operating on genuinely different scales, not
because one of them is simply wrong.

## 5. Hierarchical Bayesian G-Study

`bayesian_variance_components()` fits the same design via HMC/NUTS. This takes
longer (order of a minute on CPU for this small example) since it is drawing
posterior samples rather than solving a closed form.

In [ ]:
vc_bayes = bayesian_variance_components(
    responses,
    n_warmup=500,
    n_samples=1000,
    seed=42,
    verbose=True,
)

print("\nBayesian posterior means (and 95% credible intervals):")
ci = vc_bayes["credible_intervals"]
for key, true_key in [("subject", "sigma2_p"), ("item", "sigma2_i"), ("subject_item", "sigma2_pi")]:
    lo, hi = ci[key]
    true_val = true_vc[true_key]
    inside = lo <= true_val <= hi
    print(f"  {key:<15} mean={vc_bayes[key]:.4f}   95% CI=[{lo:.4f}, {hi:.4f}]   "
          f"true={true_val:.4f}   {'inside CI' if inside else 'OUTSIDE CI'}")

## 6. Checking Convergence

Always check these three diagnostics before trusting the posterior:

- **R-hat** (< 1.1): did the chain look stationary? (Pyro uses split R-hat, so
  this is meaningful even with a single chain.)
- **Effective sample size**: how many independent-ish samples the chain is
  worth. Low values (this happens for `sigma_pi` when there is only one
  observation per subject-item cell) mean wider Monte Carlo uncertainty on
  that estimate specifically.
- **Divergences** (should be 0): NUTS trajectories that failed. Any nonzero
  count means the posterior geometry is being explored incorrectly and the
  results should not be trusted without investigation.

In [ ]:
diag = vc_bayes["diagnostics"]
print("R-hat:      ", diag["r_hat"])
print("N_eff:      ", diag["n_eff"])
print("Divergences:", diag["divergences"])

converged = all(v < 1.1 for v in diag["r_hat"].values())
print(f"\nAll R-hat < 1.1: {converged}")

## 7. Drop-in Compatibility

Both estimators return the same four top-level keys, so everything downstream
-- `g_coefficient()`, `d_study()`, `intraclass_correlation()` -- works
unchanged regardless of which estimator produced the variance components.

In [ ]:
g_rel_anova = g_coefficient(vc_anova, n_items=n_items, type="relative")
g_rel_bayes = g_coefficient(vc_bayes, n_items=n_items, type="relative")
print(f"G_rel (ANOVA):    {g_rel_anova:.4f}")
print(f"G_rel (Bayesian): {g_rel_bayes:.4f}")

n_items_grid = [5, 10, 15, 20, 25, 30]
ds_anova = d_study(vc_anova, n_items_grid=n_items_grid, n_reps_grid=[1])
ds_bayes = d_study(vc_bayes, n_items_grid=n_items_grid, n_reps_grid=[1])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(ds_anova["n_items"], ds_anova["g_relative"], "o--", label="ANOVA")
ax.plot(ds_bayes["n_items"], ds_bayes["g_relative"], "s-", label="Bayesian")
ax.axhline(0.9, color="gray", linestyle=":", linewidth=1, label="G_rel = 0.90")
ax.set_xlabel("Number of items")
ax.set_ylabel("G_relative")
ax.set_title("D-study reliability curve: ANOVA vs. Bayesian")
ax.legend()
plt.tight_layout()
plt.show()

## 8. What This Looks Like on Real Benchmark Data

The synthetic example above is well-specified by construction -- both
estimators are fit to data generated from (close to) their own assumptions.
On real, binary benchmark data the two estimators tell noticeably different
stories, and the difference is informative rather than a bug.

Below are results (not re-run in this notebook -- each real-data Bayesian fit
takes 10-30 minutes) from two real benchmarks: `mmlu_business_ethics`
(150 models x 100 items) and SWE-Bench (134 models x 500 items).

| Component | ANOVA share (observed scale) | Bayesian share (latent scale) |
|---|---|---|
| mmlu_business_ethics: model x item interaction | 74.5% | 1.0% |
| SWE-Bench: model x item interaction | 44.9% | 0.05% |

On both datasets, ANOVA attributes a large share of total variance to
model x item interaction. The Bayesian model, on the latent (logit) scale,
finds the genuine interaction is small -- under 1% of variance on both
datasets.

**Why:** at one observation per model-item cell, every binary response
carries irreducible Bernoulli sampling noise, `p(1-p)`. ANOVA's Gaussian
model has no separate term for this noise, so it is absorbed into the
model x item interaction estimate, inflating it. The Bayesian model's
Bernoulli likelihood accounts for this noise at the point where it computes
probabilities, so its interaction estimate is not contaminated by it.

The practical takeaway: if you are running a G-study on binary right/wrong
benchmark data and see a large model x item interaction from
`variance_components()`, that is at least partly a symptom of the
observed-scale/binary-noise issue above, not necessarily evidence that
models rank inconsistently across items. `bayesian_variance_components()`
is the way to check which explanation is actually true for your data.

In [ ]:
labels = ["mmlu_business_ethics", "SWE-Bench"]
anova_share = [0.745, 0.449]
bayes_share = [0.010, 0.0005]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(x - width / 2, anova_share, width, label="ANOVA (observed scale)", color="#1565C0", alpha=0.7)
ax.bar(x + width / 2, bayes_share, width, label="Bayesian (latent scale)", color="#AD1457", alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Model x item interaction, share of total variance")
ax.set_title("ANOVA vs. Bayesian: model x item interaction on real benchmarks")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

- `variance_components()` (ANOVA) and `bayesian_variance_components()`
  (hierarchical Bayesian, HMC/NUTS) both decompose a model x item response
  matrix into the same four variance components, and share a common output
  format so downstream functions work unchanged.
- On well-specified synthetic data, both recover the true variance
  components.
- On real binary benchmark data, they can disagree substantially on how
  much variance is model x item interaction -- because ANOVA's Gaussian
  assumption does not hold for binary responses, and the Bayesian model's
  Bernoulli likelihood does not have this problem.
- Always check `diagnostics["r_hat"]`, `diagnostics["n_eff"]`, and
  `diagnostics["divergences"]` before trusting a Bayesian fit.

For the full derivation and the real-benchmark results above, see the
accompanying write-up in the `GTheory_Research` project.